# 01 - Bronze to Silver (Harborline Retail)
Owner: Data Engineering (Marcus Oyelaran). Cleans orders and customers for the net revenue model.

In [ ]:
from pyspark.sql import functions as F

# Orders: drop cancelled orders, derive net_amount (gross less discount)
orders_bronze = spark.read.table("harborline.bronze.orders_raw")
o1 = orders_bronze.filter(F.col("order_status") != "CANCELLED")
o2 = o1.withColumn("net_amount", F.col("gross_amount") - F.col("discount_amount"))
orders_clean = o2.select("order_id", "customer_id", "order_date", "gross_amount", "discount_amount", "net_amount")
orders_clean.write.mode("overwrite").saveAsTable("harborline.silver.orders_clean")


In [ ]:
%sql
CREATE OR REPLACE TABLE harborline.silver.customers_clean AS
SELECT customer_id,
       INITCAP(TRIM(full_name)) AS full_name,
       UPPER(country_code)      AS country_code,
       segment                  AS customer_segment
FROM harborline.bronze.customers_raw
WHERE is_active = true
